# **SOTA Air Pollution Forecasting: Beijing PM2.5 Prediction (GPU High Power Version)**
### **Ultra-High-Performance Feature Engineering, Target Scaling, GPU Parallelization (CUDA), & 5-Model SLSQP Ensemble**

**Competition Metric**: Root Mean Squared Error (RMSE)  
**Target Goal**: `< 13.00` (Aiming for `12.5 - 13.2` on Public/Private Leaderboard)  
**Execution Mode**: **100% GPU Accelerated (CUDA)** (In-Memory Processing - No Model Saving to Disk for Low Disk Space)  
**Output File**: `submission_gpu_ada_ensemble.csv`  

---
### **Model Training & Early Stopping Configurations**
| Model | Max Iterations / Estimators | Learning Rate | Early Stopping | Log Frequency | Disk Saving |
|---|---:|---:|---|---|---|
| **LightGBM** | `n_estimators=105000` | `0.025` | `lgb.early_stopping(100)` | Every **500** steps | ❌ Disabled |
| **XGBoost** | `n_estimators=105000` | `0.025` | `early_stopping_rounds=100` | Every **500** steps | ❌ Disabled |
| **CatBoost** | `iterations=105000` | `0.025` | `early_stopping_rounds=100` | Every **500** steps | ❌ Disabled |
| **ExtraTrees** | `n_estimators=300` | N/A | None | N/A | ❌ Disabled |
| **PyTorch ResNet-BiLSTM** | `epochs=100` (Batch Size 128) | `0.001` | `patience=10` epochs | Every **10** epochs | ❌ Disabled (In-Memory Best Preds) |


### **Step 1: Install & Import Dependencies (GPU High Power Mode)**

In [1]:
import os

# Use GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import os
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

CUDA_VISIBLE_DEVICES: 1


In [3]:
#!pip install -q lightgbm xgboost catboost scikit-learn pandas numpy torch matplotlib seaborn scipy joblib

import os
import sys
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('ggplot')
warnings.filterwarnings('ignore')

print("==================================================")
print(" STEP 1: GPU HIGH-POWER ENVIRONMENT DIAGNOSTICS")
print("==================================================")
print(f"Python Version:  {sys.version.split()[0]}")
print(f"Pandas Version:  {pd.__version__}")
print(f"NumPy Version:   {np.__version__}")
print(f"LightGBM:        {lgb.__version__}")
print(f"XGBoost:         {xgb.__version__}")
print(f"CatBoost:        {cb.__version__}")
print(f"PyTorch:         {torch.__version__}")

# Configure Hardware Acceleration for GPU High Power
USE_GPU = torch.cuda.is_available()
GPU_DEVICE = torch.device('cuda' if USE_GPU else 'cpu')
CB_TASK_TYPE = 'GPU' if USE_GPU else 'CPU'

print(f"\n[GPU STATUS] CUDA Available: {USE_GPU}")
if USE_GPU:
    print(f"[GPU MODEL]  {torch.cuda.get_device_name(0)}")
    print(f"[VRAM INFO]  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB Total VRAM")
    print(f"[EXECUTION MODE] 🚀 GPU High-Power Parallelization Enabled (In-Memory Prediction, No Disk Model Savings).")
else:
    print(f"[WARNING] GPU CUDA not detected. Falling back to multi-core CPU mode.")

 STEP 1: GPU HIGH-POWER ENVIRONMENT DIAGNOSTICS
Python Version:  3.12.3
Pandas Version:  3.0.5
NumPy Version:   2.5.1
LightGBM:        4.7.0
XGBoost:         3.3.0
CatBoost:        1.2.10
PyTorch:         2.13.0+cu130

[GPU STATUS] CUDA Available: True
[GPU MODEL]  NVIDIA RTX 6000 Ada Generation
[VRAM INFO]  47.37 GB Total VRAM
[EXECUTION MODE] 🚀 GPU High-Power Parallelization Enabled (In-Memory Prediction, No Disk Model Savings).


### **Step 2: Dataset File Verification (Auto-Detects Kaggle & Local Paths)**

In [4]:
import os
import glob
import pandas as pd

print("=" * 50)
print(" STEP 2: DATASET FILE VERIFICATION")
print("=" * 50)

def find_file(filename):
    # Search recursively inside Kaggle input directory
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        return matches[0]

    # Fallback to current directory
    if os.path.exists(filename):
        return filename

    return None

train_path = find_file("train_raw.csv")
test_path = find_file("test.csv")

for path in [train_path, test_path]:
    if path is not None:
        file_size_mb = os.path.getsize(path) / (1024 * 1024)
        df_peek = pd.read_csv(path, nrows=5)
        print(f"[FOUND] {os.path.basename(path):15s} | Size: {file_size_mb:7.2f} MB | Columns: {len(df_peek.columns)}")
        print(f"        Path: {path}")
    else:
        print(f"[WARNING] File '{path}' not found.")

 STEP 2: DATASET FILE VERIFICATION
[FOUND] train_raw.csv   | Size:   26.31 MB | Columns: 18
        Path: train_raw.csv
[FOUND] test.csv        | Size:    6.78 MB | Columns: 386
        Path: test.csv


### **Step 3: Advanced Domain Feature Engineering (Physics & Atmospheric Science)**

In [5]:
print("==================================================")
print(" STEP 3: ADVANCED DOMAIN FEATURE ENGINEERING")
print("==================================================")

WD_MAP = {
    'N': 0.0, 'NNE': 22.5, 'NE': 45.0, 'ENE': 67.5,
    'E': 90.0, 'ESE': 112.5, 'SE': 135.0, 'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0, 'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0, 'NNW': 337.5
}

POLLUTANTS = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
METEO = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
NUM_FEATURES = POLLUTANTS + METEO

def process_station_timeseries(df_station):
    df = df_station.copy()
    
    # 1. Datetime & Cyclical Trigonometric Encodings
    if 'date' not in df.columns:
        df['date'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['is_heating_season'] = df['month'].isin([11, 12, 1, 2, 3]).astype(int)
    
    # 2. Wind Direction Vector Decomposition (Wx, Wy)
    wd_degrees = df['wd'].map(WD_MAP)
    wd_rad = np.radians(wd_degrees)
    df['Wx'] = df['WSPM'] * np.sin(wd_rad)
    df['Wy'] = df['WSPM'] * np.cos(wd_rad)
    
    interp_cols = NUM_FEATURES + ['Wx', 'Wy']
    df[interp_cols] = df[interp_cols].interpolate(method='linear', limit_direction='both')
    df[interp_cols] = df[interp_cols].ffill().bfill()

    # 3. Physical Meteorology: Magnus Relative Humidity & Ventilation Index
    df['dew_point_depression'] = df['TEMP'] - df['DEWP']
    temp_c = df['TEMP']
    dew_c = df['DEWP']
    df['relative_humidity'] = 100.0 * np.exp((17.625 * dew_c)/(243.04 + dew_c) - (17.625 * temp_c)/(243.04 + temp_c))
    df['ventilation_index'] = df['WSPM'] * (df['dew_point_depression'] + 15.0)

    # 4. Atmospheric Chemistry & Coarse PM
    df['coarse_pm'] = np.maximum(df['PM10'] - df['PM2.5'], 0.0)
    df['PM25_PM10_ratio'] = df['PM2.5'] / (df['PM10'] + 1.0)
    df['coarse_ratio'] = df['coarse_pm'] / (df['PM10'] + 1.0)
    df['PM25_CO_ratio'] = df['PM2.5'] / (df['CO'] + 1.0)
    df['NO2_O3_ratio'] = df['NO2'] / (df['O3'] + 1.0)
    df['total_pollution'] = df['PM2.5'] + df['PM10'] + df['SO2'] + df['NO2'] + (df['CO'] / 1000.0) + df['O3']

    # 5. Exponential Moving Averages (EMA)
    df['PM2.5_ema_3'] = df['PM2.5'].ewm(span=3, adjust=False).mean()
    df['PM2.5_ema_6'] = df['PM2.5'].ewm(span=6, adjust=False).mean()
    df['PM2.5_ema_12'] = df['PM2.5'].ewm(span=12, adjust=False).mean()

    # 6. Lags & Multi-Step Diffs
    for lag in range(1, 25):
        df[f'PM2.5_lag_{lag}'] = df['PM2.5'].shift(lag)
        
    for k in [1, 2, 3, 4, 6, 12, 24]:
        df[f'PM2.5_diff_{k}'] = df['PM2.5'] - df['PM2.5'].shift(k)
        df[f'PM2.5_diff_{k}_ratio'] = (df['PM2.5'] - df['PM2.5'].shift(k)) / (df['PM2.5'].shift(k) + 1.0)
        
    df['PM2.5_accel'] = (df['PM2.5'] - df['PM2.5'].shift(1)) - (df['PM2.5'].shift(1) - df['PM2.5'].shift(2))

    for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'Wx', 'Wy', 'dew_point_depression', 'relative_humidity']:
        for k in [1, 3, 6]:
            df[f'{col}_diff_{k}'] = df[col] - df[col].shift(k)
            df[f'{col}_lag_{k}'] = df[col].shift(k)

    # 7. Rolling Aggregations
    for w in [3, 6, 12, 24]:
        df[f'PM2.5_roll_mean_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).mean()
        df[f'PM2.5_roll_std_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).std().fillna(0)
        df[f'PM2.5_roll_min_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).min()
        df[f'PM2.5_roll_max_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).max()
        
    df['PM2.5_roll_range_24h'] = df['PM2.5_roll_max_24h'] - df['PM2.5_roll_min_24h']
    df['PM2.5_dev_mean_6h'] = df['PM2.5'] - df['PM2.5_roll_mean_6h']
    df['PM2.5_dev_mean_24h'] = df['PM2.5'] - df['PM2.5_roll_mean_24h']
    df['PM2.5_ratio_mean_24h'] = df['PM2.5'] / (df['PM2.5_roll_mean_24h'] + 1.0)

    for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'TEMP', 'WSPM', 'dew_point_depression', 'relative_humidity']:
        for w in [3, 6, 12]:
            df[f'{col}_roll_mean_{w}h'] = df[col].rolling(w, min_periods=1).mean()
            df[f'{col}_roll_std_{w}h'] = df[col].rolling(w, min_periods=1).std().fillna(0)

    return df

print("Feature engineering logic compiled successfully!")

 STEP 3: ADVANCED DOMAIN FEATURE ENGINEERING
Feature engineering logic compiled successfully!


### **Step 4: Process Training & Test Datasets with Progress Tracking**

In [6]:
print("==================================================")
print(" STEP 4: PROCESSING SLIDING WINDOW DATASETS")
print("==================================================")

def create_train_dataset(train_csv_path):
    print(f"[TRAIN BUILD] Reading raw CSV from {train_csv_path}...")
    df_raw = pd.read_csv(train_csv_path)
    df_raw['date'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
    df_raw = df_raw.sort_values(by=['station', 'date']).reset_index(drop=True)
    
    stations = df_raw['station'].unique()
    print(f"[TRAIN BUILD] Found {len(stations)} unique air quality monitoring stations.")
    
    processed_stations = []
    for i, (station, group) in enumerate(df_raw.groupby('station')):
        df_p = process_station_timeseries(group)
        df_p['target'] = df_p['PM2.5'].shift(-1)
        processed_stations.append(df_p)
        print(f"   -> [{i+1:02d}/{len(stations):02d}] Station '{station:15s}' processed | Rows: {len(df_p):,}")
        
    df_full = pd.concat(processed_stations, ignore_index=True)
    df_train = df_full.dropna(subset=['target', 'PM2.5_lag_24']).reset_index(drop=True)
    print(f"[SUCCESS] Training dataset ready with {len(df_train):,} valid sliding window samples.")
    return df_train

def create_test_dataset(test_csv_path):
    print(f"\n[TEST BUILD] Reading test CSV from {test_csv_path}...")
    df_test_raw = pd.read_csv(test_csv_path)
    print(f"[TEST BUILD] Found {len(df_test_raw):,} test target rows.")
    
    test_rows = []
    feature_vars = ['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM']
    
    for idx, row in df_test_raw.iterrows():
        if (idx + 1) % 1000 == 0 or (idx + 1) == len(df_test_raw):
            print(f"   -> Processing Test Row [{idx+1:,} / {len(df_test_raw):,}]...", end='\r')
            
        station = row['station']
        row_id = row['id']
        
        mini_records = []
        for lag in range(24, 0, -1):
            rec = {var: row[f'{var}_lag_{lag}'] for var in feature_vars}
            mini_records.append(rec)
            
        df_mini = pd.DataFrame(mini_records)
        df_mini['station'] = station
        df_mini_p = process_station_timeseries(df_mini)
        
        last_row = df_mini_p.iloc[-1].to_dict()
        last_row['id'] = row_id
        last_row['station'] = station
        test_rows.append(last_row)
        
    df_test_proc = pd.DataFrame(test_rows)
    print(f"\n[SUCCESS] Test dataset ready with {len(df_test_proc):,} processed rows.")
    return df_test_proc

df_train = create_train_dataset(train_path)
df_test = create_test_dataset(test_path)

 STEP 4: PROCESSING SLIDING WINDOW DATASETS
[TRAIN BUILD] Reading raw CSV from train_raw.csv...
[TRAIN BUILD] Found 12 unique air quality monitoring stations.
   -> [01/12] Station 'Aotizhongxin   ' processed | Rows: 26,304
   -> [02/12] Station 'Changping      ' processed | Rows: 26,304
   -> [03/12] Station 'Dingling       ' processed | Rows: 26,304
   -> [04/12] Station 'Dongsi         ' processed | Rows: 26,304
   -> [05/12] Station 'Guanyuan       ' processed | Rows: 26,304
   -> [06/12] Station 'Gucheng        ' processed | Rows: 26,304
   -> [07/12] Station 'Huairou        ' processed | Rows: 26,304
   -> [08/12] Station 'Nongzhanguan   ' processed | Rows: 26,304
   -> [09/12] Station 'Shunyi         ' processed | Rows: 26,304
   -> [10/12] Station 'Tiantan        ' processed | Rows: 26,304
   -> [11/12] Station 'Wanliu         ' processed | Rows: 26,304
   -> [12/12] Station 'Wanshouxigong  ' processed | Rows: 26,304
[SUCCESS] Training dataset ready with 315,348 valid sliding w

### **Step 5: Feature Matrix & Station Target Encodings**

In [7]:
print("==================================================")
print(" STEP 5: FEATURE MATRIX & STATISTICAL ENCODING")
print("==================================================")

le_station = LabelEncoder()
df_train['station_cat'] = le_station.fit_transform(df_train['station'])
df_test['station_cat'] = le_station.transform(df_test['station'])

station_means = df_train.groupby('station_cat')['target'].mean().to_dict()
station_stds = df_train.groupby('station_cat')['target'].std().to_dict()
df_train['station_target_mean'] = df_train['station_cat'].map(station_means)
df_train['station_target_std'] = df_train['station_cat'].map(station_stds)
df_test['station_target_mean'] = df_test['station_cat'].map(station_means)
df_test['station_target_std'] = df_test['station_cat'].map(station_stds)

drop_cols = ['No', 'year', 'month', 'day', 'hour', 'date', 'wd', 'station', 'target', 'id']
feature_cols = [c for c in df_train.columns if c not in drop_cols]

X = df_train[feature_cols].copy().fillna(0)
y = df_train['target'].copy()
X_test = df_test[feature_cols].copy().fillna(0)

print(f"Total Input Samples (X):      {X.shape[0]:,}")
print(f"Total SOTA Feature Count (X):  {X.shape[1]:,}")
print(f"Test Matrix Shape (X_test):   {X_test.shape}")
print(f"Target Mean (PM2.5):          {y.mean():.2f} ug/m3")
print(f"Target Std  (PM2.5):          {y.std():.2f} ug/m3")
print(f"Target Range (Min to Max):    {y.min():.2f} to {y.max():.2f} ug/m3")

 STEP 5: FEATURE MATRIX & STATISTICAL ENCODING
Total Input Samples (X):      315,348
Total SOTA Feature Count (X):  224
Test Matrix Shape (X_test):   (4103, 224)
Target Mean (PM2.5):          80.48 ug/m3
Target Std  (PM2.5):          80.42 ug/m3
Target Range (Min to Max):    2.00 to 999.00 ug/m3


### **Step 6: PyTorch Deep ResNet-1D BiLSTM Model Architecture (CUDA Enabled)**

In [8]:
print("==================================================")
print(" STEP 6: PYTORCH DEEP RESNET-1D BILSTM MODEL")
print("==================================================")

class ResNet1DBlock(nn.Module):
    def __init__(self, channels):
        super(ResNet1DBlock, self).__init__()
        self.fc1 = nn.Linear(channels, channels)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(channels, channels)
        
    def forward(self, x):
        residual = x
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        return self.relu(out + residual)

class DeepResNet_BiLSTM(nn.Module):
    def __init__(self, input_dim):
        super(DeepResNet_BiLSTM, self).__init__()
        self.in_proj = nn.Linear(input_dim, 256)
        self.res1 = ResNet1DBlock(256)
        self.res2 = ResNet1DBlock(256)
        
        self.bilstm = nn.LSTM(256, 128, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        h = self.in_proj(x)
        h = self.res1(h)
        h = self.res2(h)
        h = h.unsqueeze(1)
        out, _ = self.bilstm(h)
        out = out.squeeze(1)
        out = self.head(out)
        return out.squeeze(-1)

sample_net = DeepResNet_BiLSTM(X.shape[1]).to(GPU_DEVICE)
total_params = sum(p.numel() for p in sample_net.parameters() if p.requires_grad)
print(f"DeepResNet_BiLSTM initialized on [{GPU_DEVICE}] | Trainable Parameters: {total_params:,}")

 STEP 6: PYTORCH DEEP RESNET-1D BILSTM MODEL
DeepResNet_BiLSTM initialized on [cuda] | Trainable Parameters: 1,152,513


### **Step 7: 5-Fold GPU High-Power Cross-Validation Training & Live Evaluation (In-Memory Processing)**

In [ ]:
print("==================================================")
print(" STEP 7: 5-FOLD GPU HIGH-POWER MODEL TRAINING")
print("==================================================")

# Note: Model disk saving disabled to conserve disk space. All OOF & test predictions held in-memory.
oof_lgb = np.zeros(len(df_train))
oof_xgb = np.zeros(len(df_train))
oof_cb  = np.zeros(len(df_train))
oof_et  = np.zeros(len(df_train))
oof_nn  = np.zeros(len(df_train))

preds_lgb = np.zeros(len(df_test))
preds_xgb = np.zeros(len(df_test))
preds_cb  = np.zeros(len(df_test))
preds_et  = np.zeros(len(df_test))
preds_nn  = np.zeros(len(df_test))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n" + "="*50)
    print(f"   >>> RUNNING FOLD {fold+1} / 5 (GPU HIGH-POWER MODE) <<<")
    print("="*50)
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    print(f"Train samples: {len(X_tr):,} | Val samples: {len(X_va):,}")
    
    # 1. LIGHTGBM (GPU Accelerated - Logging every 500 steps)
    print(f"\n[FOLD {fold+1}] [1/5] Training LightGBM (n_estimators=105000, lr=0.025, early_stopping=100, log_period=500)...")
    lgb_kwargs = {
        'n_estimators': 105000,
        'learning_rate': 0.025,
        'num_leaves': 63,
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.65,
        'objective': 'regression',
        'metric': 'rmse',
        'random_state': 42 + fold,
        'n_jobs': 8
    }
    if USE_GPU:
        try:
            lgb_kwargs['device'] = 'gpu'
        except Exception:
            pass
            
    model_lgb = lgb.LGBMRegressor(**lgb_kwargs)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(period=500)
        ]
    )
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    preds_lgb += model_lgb.predict(X_test) / 5.0
    lgb_score = np.sqrt(mean_squared_error(y_va, oof_lgb[val_idx]))
    print(f"==> Fold {fold+1} LightGBM RMSE: {lgb_score:.4f} (Best Iteration: {model_lgb.best_iteration_})")
    
    # 2. XGBOOST (GPU Accelerated - Logging every 500 steps)
    print(f"\n[FOLD {fold+1}] [2/5] Training XGBoost (n_estimators=105000, lr=0.025, early_stopping_rounds=100, verbose=500)...")
    xgb_kwargs = {
        'n_estimators': 105000,
        'learning_rate': 0.025,
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.65,
        'gamma': 0.1,
        'reg_alpha': 0.5,
        'reg_lambda': 1.5,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': 42 + fold,
        'n_jobs': -1,
        'early_stopping_rounds': 100
    }
    if USE_GPU:
        xgb_kwargs['tree_method'] = 'hist'
        xgb_kwargs['device'] = 'cuda'
    else:
        xgb_kwargs['tree_method'] = 'hist'
        
    model_xgb = xgb.XGBRegressor(**xgb_kwargs)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=500)
    oof_xgb[val_idx] = model_xgb.predict(X_va)
    preds_xgb += model_xgb.predict(X_test) / 5.0
    xgb_score = np.sqrt(mean_squared_error(y_va, oof_xgb[val_idx]))
    print(f"==> Fold {fold+1} XGBoost RMSE:  {xgb_score:.4f} (Best Iteration: {model_xgb.best_iteration})")
    
    # 3. CATBOOST (GPU Accelerated - Logging every 500 steps)
    print(f"\n[FOLD {fold+1}] [3/5] Training CatBoost (iterations=105000, lr=0.025, early_stopping_rounds=100, verbose=500)...")
    

    model_cb = cb.CatBoostRegressor(
        iterations=105000,   # 0.025
        learning_rate=0.025, #for 8 i got 15.3976 2hr
        depth=8,
        #border_count=64,             # ⚡ Reduces VRAM usage by 4x (STOPS PCIe swapping!)
        #gpu_ram_part=0.7,            # ⚡ Restricts VRAM allocation so it fits inside free GPU RAM
        loss_function='RMSE',
        eval_metric='RMSE',
        task_type="GPU",
        devices="0",    #0
        thread_count=-1,
        random_seed=42 + fold,
        verbose=500
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)
    oof_cb[val_idx] = model_cb.predict(X_va)
    preds_cb += model_cb.predict(X_test) / 5.0
    cb_score = np.sqrt(mean_squared_error(y_va, oof_cb[val_idx]))
    print(f"==> Fold {fold+1} CatBoost RMSE: {cb_score:.4f} (Best Iteration: {model_cb.get_best_iteration()})")
    
    # 4. EXTRATREES (300 Trees Multi-Threaded CPU)
    print(f"\n[FOLD {fold+1}] [4/5] Training ExtraTrees (n_estimators=300, max_depth=16)...")
    model_et = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=16,
        min_samples_split=5,
        n_jobs=-1,
        random_state=42 + fold
    )
    model_et.fit(X_tr, y_tr)
    oof_et[val_idx] = model_et.predict(X_va)
    preds_et += model_et.predict(X_test) / 5.0
    et_score = np.sqrt(mean_squared_error(y_va, oof_et[val_idx]))
    print(f"==> Fold {fold+1} ExtraTrees RMSE:{et_score:.4f}")
    
    # 5. PYTORCH RESNET-1D BILSTM (In-Memory Processing - No Disk Weight Saving)
    print(f"\n[FOLD {fold+1}] [5/5] Training PyTorch ResNet-BiLSTM (epochs=100, batch_size=128, patience=10)...")
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    
    X_tr_sc = scaler_x.fit_transform(X_tr)
    X_va_sc = scaler_x.transform(X_va)
    X_te_sc = scaler_x.transform(X_test)
    
    y_tr_sc = scaler_y.fit_transform(y_tr.values.reshape(-1, 1)).flatten()
    y_va_sc = scaler_y.transform(y_va.values.reshape(-1, 1)).flatten()
    
    ds_tr = TensorDataset(torch.tensor(X_tr_sc, dtype=torch.float32), torch.tensor(y_tr_sc, dtype=torch.float32))
    ds_va = TensorDataset(torch.tensor(X_va_sc, dtype=torch.float32), torch.tensor(y_va_sc, dtype=torch.float32))
    
    dl_tr = DataLoader(ds_tr, batch_size=128, shuffle=True, pin_memory=True if USE_GPU else False)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False, pin_memory=True if USE_GPU else False)
    
    net = DeepResNet_BiLSTM(X_tr.shape[1]).to(GPU_DEVICE)
    optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    best_preds = None
    best_test_preds = None
    patience = 10
    patience_counter = 0
    
    for epoch in range(1, 101):
        net.train()
        running_loss = 0.0
        for bx, by in dl_tr:
            bx, by = bx.to(GPU_DEVICE), by.to(GPU_DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(bx), by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)
            
        train_epoch_loss = running_loss / len(ds_tr)
        
        net.eval()
        val_preds_list = []
        with torch.no_grad():
            for bx, by in dl_va:
                bx = bx.to(GPU_DEVICE)
                val_preds_list.append(net(bx).cpu().numpy())
        val_preds_sc = np.concatenate(val_preds_list)
        val_preds_raw = scaler_y.inverse_transform(val_preds_sc.reshape(-1, 1)).flatten()
        val_rmse = np.sqrt(mean_squared_error(y_va, val_preds_raw))
        
        scheduler.step(val_rmse)
        
        if val_rmse < best_loss:
            best_loss = val_rmse
            best_preds = val_preds_raw
            patience_counter = 0
            
            # Compute best test set predictions in-memory without saving checkpoint files
            test_tensor = torch.tensor(X_te_sc, dtype=torch.float32).to(GPU_DEVICE)
            te_sc = net(test_tensor).detach().cpu().numpy()
            best_test_preds = scaler_y.inverse_transform(te_sc.reshape(-1, 1)).flatten()
        else:
            patience_counter += 1
            
        if epoch % 10 == 0 or patience_counter >= patience:
            print(f"   Epoch [{epoch:03d}/100] Train Loss: {train_epoch_loss:.4f} | Val RMSE: {val_rmse:.4f} | Best RMSE: {best_loss:.4f} (Patience: {patience_counter}/{patience})")
            
        if patience_counter >= patience:
            print(f"   --> [EARLY STOPPING] PyTorch NN stopped early at Epoch {epoch}! Best Val RMSE: {best_loss:.4f}")
            break
            
    oof_nn[val_idx] = best_preds
    preds_nn += best_test_preds / 5.0
    print(f"==> Fold {fold+1} PyTorch ResNet RMSE: {best_loss:.4f}")

 STEP 7: 5-FOLD GPU HIGH-POWER MODEL TRAINING

   >>> RUNNING FOLD 1 / 5 (GPU HIGH-POWER MODE) <<<
Train samples: 252,278 | Val samples: 63,070

[FOLD 1] [1/5] Training LightGBM (n_estimators=105000, lr=0.025, early_stopping=100, log_period=500)...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 53284
[LightGBM] [Info] Number of data points in the train set: 252278, number of used features: 224
[LightGBM] [Info] Using GPU Device: NVIDIA RTX 6000 Ada Generation, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 223 dense feature groups (53.89 MB) transferred to GPU in 0.010476 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 80.472080
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

### **Step 8: Mathematical SLSQP Optimal Blending & Score Summary**

In [ ]:
print("==================================================")
print(" STEP 8: OUT-OF-FOLD EVALUATION & SLSQP BLENDING")
print("==================================================")

oof_list = [oof_lgb, oof_xgb, oof_cb, oof_et, oof_nn]
test_preds_list = [preds_lgb, preds_xgb, preds_cb, preds_et, preds_nn]
model_names = ['LightGBM', 'XGBoost', 'CatBoost', 'ExtraTrees', 'PyTorch ResNet']

print("--- Individual Model Out-Of-Fold (OOF) Scores ---")
for name, oof in zip(model_names, oof_list):
    score = np.sqrt(mean_squared_error(y, oof))
    print(f"   {name:25s} -> OOF RMSE: {score:.5f}")

def optimize_blend_weights(oof_preds_list, y_true):
    num_models = len(oof_preds_list)
    def loss_func(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blend = sum(w[i] * oof_preds_list[i] for i in range(num_models))
        return np.sqrt(mean_squared_error(y_true, blend))
    
    init_weights = np.ones(num_models) / num_models
    bounds = [(0.0, 1.0)] * num_models
    constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
    res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x / np.sum(res.x)

print("\n--- Solving SLSQP Convex Optimization for Ensemble Weights ---")
opt_w = optimize_blend_weights(oof_list, y.values)
for name, w in zip(model_names, opt_w):
    print(f"   Weight [{name:22s}] = {w:6.4f} ({w*100:5.2f}%)")

final_oof = sum(opt_w[i] * oof_list[i] for i in range(len(oof_list)))
final_rmse = np.sqrt(mean_squared_error(y, final_oof))

print(f"\n" + "*"*55)
print(f"*** OPTIMAL SOTA HYBRID ENSEMBLE OOF RMSE: {final_rmse:.5f} ***")
print(f"" + "*"*55)

final_test_preds = sum(opt_w[i] * test_preds_list[i] for i in range(len(test_preds_list)))
final_test_preds = np.clip(final_test_preds, a_min=0.0, a_max=None)
print("[PIPELINE COMPLETE] In-memory ensemble evaluation completed.")

### **Step 9: Save Kaggle Submission CSV**

In [ ]:
print("==================================================")
print(" STEP 9: FINAL SUBMISSION FILE GENERATION")
print("==================================================")

sub_df = pd.DataFrame({
    'id': df_test['id'],
    'PM2.5': final_test_preds
})

sub_filename = 'submission_gpu_ada_ensemble.csv'
sub_df.to_csv(sub_filename, index=False)
print(f"[SUCCESS] Saved {len(sub_df):,} predictions to '{sub_filename}'")
print(f"Submission Stats:")
print(f"   Minimum Predicted PM2.5: {sub_df['PM2.5'].min():.2f} ug/m3")
print(f"   Maximum Predicted PM2.5: {sub_df['PM2.5'].max():.2f} ug/m3")
print(f"   Mean Predicted PM2.5:    {sub_df['PM2.5'].mean():.2f} ug/m3")
print(f"   Null Count:              {sub_df['PM2.5'].isnull().sum()}")
display(sub_df.head(10))

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(sub_filename)